# Drax 구동 확인

참고: CLAUDE.md 3.2절, `docs/setup_drax_whisper_llada.md` 2절 (할 일 3)

확인할 것: ① 정상 전사 ② GPU 메모리·클립당 시간 ③ 중간 단계 추출

**별도 Colab 세션에서 실행한다.**

In [ ]:
# 공통 설정: GitHub에서 코드 받기 + Drive 연결
# Drive에는 데이터·모델 캐시·결과를 둔다 (세션이 끊겨도 남음)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/diffusion-asr-hallucination'
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/alsgur4082/diffusion-asr-hallucination.git {REPO}
sys.path.insert(0, REPO)

DRIVE = '/content/drive/MyDrive/dah'   # 데이터·캐시·결과 루트
DATA = f'{DRIVE}/data'
RESULTS = f'{DRIVE}/results'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'  # 모델 import 전에 설정해야 적용됨
os.makedirs(RESULTS, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab torch 버전 확인. 2.7.x가 아니고 설치·실행이 실패하면 아래 주석을 풀어 고정 설치
import torch; print(torch.__version__, torch.version.cuda)
# !pip install -q torch==2.7.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128
!git clone -q https://github.com/aiola-lab/drax.git /content/drax 2>/dev/null || true
!pip install -q -e /content/drax
!pip install -q jiwer soundfile scipy
# import가 실패하면: 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
# 스모크 테스트용 소량 데이터 (LibriSpeech test-clean 5개)
from src.data.download import librispeech
ls_paths, ls_refs = librispeech(DATA, 'test-clean', n=5)
print(len(ls_paths), ls_paths[0], ls_refs[0])

## 1. 정상 전사 + 메모리·시간

In [ ]:
import time, torch
from drax import Transcriber
from src.runners.base import DecodeConfig, TranscriptionResult, save_results
from src.metrics.basic import wer

cfg = DecodeConfig(model='aiola/drax-v1', language='en', temperature=1e-2, num_steps=32, seed=0)
torch.manual_seed(cfg.seed)
asr = Transcriber(model_path=cfg.model)
torch.cuda.reset_peak_memory_stats()

res = []
for p in ls_paths:
    t0 = time.time()
    out = asr.transcribe(p, language=cfg.language, sampling_steps=cfg.num_steps, temperature=cfg.temperature)
    res.append(TranscriptionResult(audio_path=p, transcript=out[0].transcript, elapsed_sec=time.time() - t0, config=cfg))

for r, ref in zip(res, ls_refs):
    print('REF:', ref); print('HYP:', r.transcript); print(f'{r.elapsed_sec:.2f}s'); print()
print(wer(ls_refs, [r.transcript for r in res]))
print(f'최대 GPU 메모리: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
save_results(res, f'{RESULTS}/drax_smoke.jsonl')

## 2. 중간 단계 추출 (RQ2)

`Transcriber`는 중간 단계를 돌려주지 않으므로 solver를 직접 부른다. 코드를 읽고 쓴 초안이라 실제 동작을 여기서 확인한다.

In [ ]:
torch.manual_seed(0)
steps = cfg.num_steps
x_init, preserve_mask, audio_cache = asr._build_batch([ls_paths[0]], ['en'], None)
traj = asr.solver.sample(
    x_init=x_init,
    step_size=1 / steps,
    time_grid=torch.linspace(0, 1, steps + 1, device=x_init.device),
    return_intermediates=True,
    preserve_mask=preserve_mask,
    temperature=cfg.temperature,
    **audio_cache,
)
print('궤적 shape (시점, 배치, 길이):', tuple(traj.shape))
for i, t in enumerate(traj):
    print(i, asr.processor.batch_decode(t, skip_special_tokens=True)[0][:120])